In [11]:
# 1. Import required libraries

from pathlib import Path
# defaultdict is used to group files by match_id
# Counter is used to count occurrences
from collections import defaultdict, Counter
import polars as pl

In [46]:
# 2. Define project paths

#get the current notebook directory
CURRENT_DIR=Path.cwd()

PROJECT_ROOT = CURRENT_DIR.parent

DATA_ROOT = PROJECT_ROOT / "tennis_data"

# Define the directory containing extracted data
EXTRACT_ROOT = DATA_ROOT / "extracted"

print("CURRENT_DIR:",CURRENT_DIR)
print("PROJECT_ROOT:",PROJECT_ROOT)
print("DATA_ROOT",DATA_ROOT)
print("EXTRACT_ROOT:", EXTRACT_ROOT)


CURRENT_DIR: /Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/notebooks
PROJECT_ROOT: /Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis
DATA_ROOT /Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data
EXTRACT_ROOT: /Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted


In [16]:
# 3. Find all Round Parquet files

# Search inside all snapshot folders for Round Parquet files
round_files = sorted(
    EXTRACT_ROOT.glob(
        "*/round_*.parquet"
    )
)



print("Number of Round files:",len(round_files))

Number of Round files: 19283


In [17]:
# 4. Display a few Round files

for file in round_files[:10]:
    print(file)

/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/round_11998445.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/round_11998446.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/round_11998447.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/round_11998448.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/round_11998449.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/round_11998450.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/round_11998451.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/

In [18]:
# 5. Group Round files by match_id


# Create a dictionary where each match_id
# will contain a list of its corresponding files
match_files = defaultdict(list)



for file in round_files:
    match_id = file.stem.replace("round_","")

    match_files[match_id].append(file)

print("Total unique match IDs:",len(match_files))


Total unique match IDs: 9243


In [19]:
# 6. Find match IDs with multiple snapshots
duplicate_matches = {
    match_id: files
    for match_id, files in match_files.items()
    if len(files) > 1
}


print("Number of match IDs with multiple snapshots:",len(duplicate_matches))

Number of match IDs with multiple snapshots: 8778


In [21]:
# 7. Select one match ID for snapshot analysis

# Select the first match ID that appears
# in multiple snapshots

sample_match_id = next(
    iter(duplicate_matches)
)

# Get all Round files belonging to this match

files = duplicate_matches[
    sample_match_id
]

print("Analyzing Match ID:",sample_match_id)

print("Number of snapshots:",len(files))

Analyzing Match ID: 11998445
Number of snapshots: 2


In [22]:
# 8. Read all snapshots of the selected match
# Read all Parquet files belonging to this match

dfs = [
    pl.read_parquet(file)
    for file in files
]

In [23]:
# 9. Display the snapshots
# Display each snapshot with its snapshot date

for file, df in zip(files, dfs):

    print("=" * 50)

    print("Snapshot:",file.parent.name)

    print(df)

Snapshot: 20240201
shape: (1, 5)
┌──────────┬──────────┬─────────────┬─────────────┬────────────────┐
│ match_id ┆ round_id ┆ name        ┆ slug        ┆ cup_round_type │
│ ---      ┆ ---      ┆ ---         ┆ ---         ┆ ---            │
│ i64      ┆ i64      ┆ str         ┆ str         ┆ i64            │
╞══════════╪══════════╪═════════════╪═════════════╪════════════════╡
│ 11998445 ┆ 5        ┆ Round of 16 ┆ round-of-16 ┆ 8              │
└──────────┴──────────┴─────────────┴─────────────┴────────────────┘
Snapshot: 20240202
shape: (1, 5)
┌──────────┬──────────┬─────────────┬─────────────┬────────────────┐
│ match_id ┆ round_id ┆ name        ┆ slug        ┆ cup_round_type │
│ ---      ┆ ---      ┆ ---         ┆ ---         ┆ ---            │
│ i64      ┆ i64      ┆ str         ┆ str         ┆ i64            │
╞══════════╪══════════╪═════════════╪═════════════╪════════════════╡
│ 11998445 ┆ 5        ┆ Round of 16 ┆ round-of-16 ┆ 8              │
└──────────┴──────────┴─────────────┴

In [24]:
# 10. Check the schema of all Round files
# Store the schemas of all Round files

schemas = Counter()

for file in round_files:

    df = pl.read_parquet(file)

    # Convert schema to a tuple so it can be counted

    schema_tuple = tuple(
        df.schema.items()
    )

    schemas[schema_tuple] += 1


print("Number of different schemas:",len(schemas))

Number of different schemas: 2


In [25]:
# 11. Display the different schemas
# Display each unique schema
# and the number of files using that schema

for i, (schema, count) in enumerate(schemas.items(), start=1):

    print("=" * 60)

    print(f"Schema {i}")
    print(f"Number of files: {count}")

    print("-" * 60)

    for column, dtype in schema:
        print(f"{column} -> {dtype}")

Schema 1
Number of files: 16656
------------------------------------------------------------
match_id -> Int64
round_id -> Int64
name -> String
slug -> String
cup_round_type -> Int64
Schema 2
Number of files: 2627
------------------------------------------------------------
match_id -> Int64
round_id -> Int64
name -> String
slug -> String
cup_round_type -> Null


In [26]:
# 12. Find files where cup_round_type has Null data type
# Store files where cup_round_type has Null data type

null_type_files = []

for file in round_files:
    df = pl.read_parquet(file)

    # Check the data type of cup_round_type
    if df.schema["cup_round_type"] == pl.Null:

        null_type_files.append(file)



print("Files with Null cup_round_type:",len(null_type_files))


for file in null_type_files[:10]:
    print(file)

Files with Null cup_round_type: 2627
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240202/round_12027640.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240202/round_12027641.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240202/round_12027643.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240202/round_12027644.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240202/round_12027645.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240202/round_12027646.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240202/round_12027647.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis

In [27]:
# 13. Inspect a sample file with Null cup_round_type

# Select the first file where cup_round_type has Null data type
sample_null_file = null_type_files[0]


print("Sample file:")
print(sample_null_file)




sample_null_df = pl.read_parquet(sample_null_file)



print("\nData:")
print(sample_null_df)


print("\nSchema:")
print(sample_null_df.schema)

Sample file:
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240202/round_12027640.parquet

Data:
shape: (1, 5)
┌──────────┬──────────┬───────────────────────┬───────────────────────┬────────────────┐
│ match_id ┆ round_id ┆ name                  ┆ slug                  ┆ cup_round_type │
│ ---      ┆ ---      ┆ ---                   ┆ ---                   ┆ ---            │
│ i64      ┆ i64      ┆ str                   ┆ str                   ┆ null           │
╞══════════╪══════════╪═══════════════════════╪═══════════════════════╪════════════════╡
│ 12027640 ┆ 1        ┆ Qualification round 1 ┆ qualification-round-1 ┆ null           │
└──────────┴──────────┴───────────────────────┴───────────────────────┴────────────────┘

Schema:
Schema({'match_id': Int64, 'round_id': Int64, 'name': String, 'slug': String, 'cup_round_type': Null})


In [28]:
# 14. Check other snapshots of the same match

# Get the match_id from the sample file
sample_match_id = sample_null_df["match_id"][0]

print("Match ID:", sample_match_id)


# Check whether this match appears in multiple snapshots

if str(sample_match_id) in match_files:

    files = match_files[str(sample_match_id)]

    print("Number of snapshots:", len(files))

    # Read and display all snapshots of this match

    for file in files:

        df = pl.read_parquet(file)

        print("=" * 60)
        print("Snapshot:", file.parent.name)
        print(df)

else:

    print("No other snapshots found.")

Match ID: 12027640
Number of snapshots: 2
Snapshot: 20240202
shape: (1, 5)
┌──────────┬──────────┬───────────────────────┬───────────────────────┬────────────────┐
│ match_id ┆ round_id ┆ name                  ┆ slug                  ┆ cup_round_type │
│ ---      ┆ ---      ┆ ---                   ┆ ---                   ┆ ---            │
│ i64      ┆ i64      ┆ str                   ┆ str                   ┆ null           │
╞══════════╪══════════╪═══════════════════════╪═══════════════════════╪════════════════╡
│ 12027640 ┆ 1        ┆ Qualification round 1 ┆ qualification-round-1 ┆ null           │
└──────────┴──────────┴───────────────────────┴───────────────────────┴────────────────┘
Snapshot: 20240203
shape: (1, 5)
┌──────────┬──────────┬───────────────────────┬───────────────────────┬────────────────┐
│ match_id ┆ round_id ┆ name                  ┆ slug                  ┆ cup_round_type │
│ ---      ┆ ---      ┆ ---                   ┆ ---                   ┆ ---            │
│ 

In [29]:
# 15. Analyze cup_round_type values

# Read all Round files and collect cup_round_type values
cup_round_values = []

for file in round_files:

    df = pl.read_parquet(file)

    cup_round_values.extend(
        df["cup_round_type"].to_list()
    )


# Count total values

print("Total cup_round_type values:",len(cup_round_values))


# Count missing values

print(
    "Missing cup_round_type values:",
    sum(value is None for value in cup_round_values)
)


# Display unique non-null values

unique_values = sorted(
    set(
        value
        for value in cup_round_values
        if value is not None
    )
)

print(
    "Unique non-null cup_round_type values:"
)

print(
    unique_values
)

Total cup_round_type values: 19283
Missing cup_round_type values: 2627
Unique non-null cup_round_type values:
[1, 2, 4, 8, 16]


In [30]:
# 16. Define the standard schema for Round data

# Use one consistent data type for each Round column.
# cup_round_type is Int64 because non-null files use Int64.
# Null values will be preserved during casting.

round_schema = {
    "match_id": pl.Int64,
    "round_id": pl.Int64,
    "name": pl.String,
    "slug": pl.String,
    "cup_round_type": pl.Int64
}

print("Standard Round schema:")
print(round_schema)

Standard Round schema:
{'match_id': Int64, 'round_id': Int64, 'name': String, 'slug': String, 'cup_round_type': Int64}


In [31]:
# 16. Read and standardize all Round Parquet files

# This list will store the processed DataFrames
round_frames = []


# Process every Round Parquet file

for file in round_files:

    df = pl.read_parquet(file)


    snapshot_date = file.parent.name

    df = df.with_columns(
        pl.lit(snapshot_date)
        .str.strptime(
            pl.Date,
            "%Y%m%d"
        )
        .alias("snapshot_date")
    )



    # Standardize column data types
    #
    # strict=False allows existing Null values
    # in cup_round_type to remain Null.
   

    df = df.cast(
        round_schema,
        strict=False
    )


    # Add the processed DataFrame to the list


    round_frames.append(df)




print("Number of processed Round files:",len(round_frames))

Number of processed Round files: 19283


In [32]:
# 17. Concatenate all Round DataFrames
# Combine all processed Round DataFrames vertically.
# Each DataFrame represents a Round record from a snapshot.

round_snapshot = pl.concat(
    round_frames,
    how="vertical"
)


print("Final Round dataset shape:",round_snapshot.shape)

Final Round dataset shape: (19283, 6)


In [33]:
# 18. Check missing values


# Count missing values in each column

null_counts = round_snapshot.null_count()

print(null_counts)

shape: (1, 6)
┌──────────┬──────────┬──────┬──────┬────────────────┬───────────────┐
│ match_id ┆ round_id ┆ name ┆ slug ┆ cup_round_type ┆ snapshot_date │
│ ---      ┆ ---      ┆ ---  ┆ ---  ┆ ---            ┆ ---           │
│ u32      ┆ u32      ┆ u32  ┆ u32  ┆ u32            ┆ u32           │
╞══════════╪══════════╪══════╪══════╪════════════════╪═══════════════╡
│ 0        ┆ 0        ┆ 0    ┆ 0    ┆ 2627           ┆ 0             │
└──────────┴──────────┴──────┴──────┴────────────────┴───────────────┘


In [34]:
# 19. Check duplicated Round records

# Check duplicated rows based on all columns except snapshot_date

duplicate_count = (
    round_snapshot
    .is_duplicated()
    .sum()
)

print("Number of duplicated rows:",duplicate_count)

Number of duplicated rows: 0


In [35]:
# 20. Check repeated match_id across snapshots

match_snapshot_count = (
    round_snapshot
    .group_by("match_id")
    .agg(
        pl.col("snapshot_date")
        .n_unique()
        .alias("number_of_snapshots")
    )
)


# Show matches with more than one snapshot

multiple_snapshots = (
    match_snapshot_count
    .filter(
        pl.col("number_of_snapshots") > 1
    )
)


print(
    "Match IDs with multiple snapshots:",
    multiple_snapshots.shape[0]
)


multiple_snapshots.head(10)

Match IDs with multiple snapshots: 8778


match_id,number_of_snapshots
i64,u32
12168587,2
12152331,2
12086652,2
12027666,2
12184822,2
12084824,2
12160167,2
12156231,2
12128185,2


In [36]:
# 21. Check if Round information changes between snapshots

round_changes = (
    round_snapshot
    .group_by("match_id")
    .agg(
        pl.col("round_id").n_unique().alias("different_round_ids"),
        pl.col("name").n_unique().alias("different_names"),
        pl.col("slug").n_unique().alias("different_slugs"),
        pl.col("cup_round_type").n_unique().alias("different_cup_types")
    )
)


# Find matches where Round information changed

changed_matches = (
    round_changes
    .filter(
        (pl.col("different_round_ids") > 1)
        |
        (pl.col("different_names") > 1)
        |
        (pl.col("different_slugs") > 1)
        |
        (pl.col("different_cup_types") > 1)
    )
)


print(
    "Number of matches with Round changes:",
    changed_matches.shape[0]
)


changed_matches.head(10)

Number of matches with Round changes: 125


match_id,different_round_ids,different_names,different_slugs,different_cup_types
i64,u32,u32,u32,u32
12112706,1,2,2,1
12111700,1,2,2,1
12114293,1,2,2,1
12113638,1,2,2,1
12088090,1,2,2,1
12115070,1,2,2,1
12114412,1,2,2,1
12109842,1,2,2,1
12088078,1,2,2,1


In [37]:
# 22. Inspect one match with Round changes

# Select one suspicious match_id

sample_changed_match = changed_matches["match_id"][0]


print(
    "Analyzing match_id:",
    sample_changed_match
)


# Show all snapshots of this match

sample_data = (
    round_snapshot
    .filter(
        pl.col("match_id") == sample_changed_match
    )
    .sort("snapshot_date")
)


print(sample_data)

Analyzing match_id: 12112706
shape: (2, 6)
┌──────────┬──────────┬───────────────┬───────────────┬────────────────┬───────────────┐
│ match_id ┆ round_id ┆ name          ┆ slug          ┆ cup_round_type ┆ snapshot_date │
│ ---      ┆ ---      ┆ ---           ┆ ---           ┆ ---            ┆ ---           │
│ i64      ┆ i64      ┆ str           ┆ str           ┆ i64            ┆ date          │
╞══════════╪══════════╪═══════════════╪═══════════════╪════════════════╪═══════════════╡
│ 12112706 ┆ 27       ┆ Quarterfinal  ┆ quarterfinal  ┆ 4              ┆ 2024-02-29    │
│ 12112706 ┆ 27       ┆ Quarterfinals ┆ quarterfinals ┆ 4              ┆ 2024-03-01    │
└──────────┴──────────┴───────────────┴───────────────┴────────────────┴───────────────┘


In [38]:
# 23. Count name and slug variations

# Find all matches where only name or slug changes
# while round_id and cup_round_type stay the same

name_slug_changes = (
    round_changes
    .filter(
        (pl.col("different_names") > 1)
        &
        (pl.col("different_round_ids") == 1)
        &
        (pl.col("different_cup_types") <= 1)
    )
)

print("Matches with only name/slug changes:",name_slug_changes.shape[0])


Matches with only name/slug changes: 125


In [40]:
# 24. Find different names for the same round_id

round_name_variations = (
    round_snapshot
    .group_by("round_id")
    .agg(
        pl.col("name")
        .unique()
        .alias("different_names"),

        pl.col("slug")
        .unique()
        .alias("different_slugs")
    )
    .filter(
        pl.col("different_names")
        .list.len() > 1
    )
)


print("Round IDs with different names:",round_name_variations)

Round IDs with different names: shape: (2, 3)
┌──────────┬─────────────────────────────────┬─────────────────────────────────┐
│ round_id ┆ different_names                 ┆ different_slugs                 │
│ ---      ┆ ---                             ┆ ---                             │
│ i64      ┆ list[str]                       ┆ list[str]                       │
╞══════════╪═════════════════════════════════╪═════════════════════════════════╡
│ 27       ┆ ["Quarterfinal", "Quarterfinal… ┆ ["quarterfinal", "quarterfinal… │
│ 28       ┆ ["Semifinal", "Semifinals"]     ┆ ["semifinal", "semifinals"]     │
└──────────┴─────────────────────────────────┴─────────────────────────────────┘


In [41]:
# 25. Standardize Round names and slugs

# Replace plural forms with standard singular forms

round_snapshot = round_snapshot.with_columns(

    pl.col("name")
    .replace({
        "Quarterfinals": "Quarterfinal",
        "Semifinals": "Semifinal"
    }),

    pl.col("slug")
    .replace({
        "quarterfinals": "quarterfinal",
        "semifinals": "semifinal"
    })

)


# Check the result

print(
    round_snapshot
    .select(
        ["name", "slug"]
    )
    .unique()
)

shape: (9, 2)
┌───────────────────────┬───────────────────────┐
│ name                  ┆ slug                  │
│ ---                   ┆ ---                   │
│ str                   ┆ str                   │
╞═══════════════════════╪═══════════════════════╡
│ Round of 16           ┆ round-of-16           │
│ Round of 64           ┆ round-of-64           │
│ Final                 ┆ final                 │
│ Quarterfinal          ┆ quarterfinal          │
│ Round of 128          ┆ round-of-128          │
│ Semifinal             ┆ semifinal             │
│ Qualification round 1 ┆ qualification-round-1 │
│ Round of 32           ┆ round-of-32           │
│ Qualification round 2 ┆ qualification-round-2 │
└───────────────────────┴───────────────────────┘


In [42]:
# 26.Final validation of round names

final_check = (
    round_snapshot
    .group_by("round_id")
    .agg(
        pl.col("name")
        .n_unique()
        .alias("unique_names"),

        pl.col("slug")
        .n_unique()
        .alias("unique_slugs")
    )
)

print(final_check)

shape: (9, 3)
┌──────────┬──────────────┬──────────────┐
│ round_id ┆ unique_names ┆ unique_slugs │
│ ---      ┆ ---          ┆ ---          │
│ i64      ┆ u32          ┆ u32          │
╞══════════╪══════════════╪══════════════╡
│ 6        ┆ 1            ┆ 1            │
│ 27       ┆ 1            ┆ 1            │
│ 1        ┆ 1            ┆ 1            │
│ 64       ┆ 1            ┆ 1            │
│ 28       ┆ 1            ┆ 1            │
│ 5        ┆ 1            ┆ 1            │
│ 2        ┆ 1            ┆ 1            │
│ 29       ┆ 1            ┆ 1            │
│ 32       ┆ 1            ┆ 1            │
└──────────┴──────────────┴──────────────┘


In [43]:
# 27. Save cleaned Round dataset

# Define output path

clean_path = DATA_ROOT / "Data"

# Create directory if it does not exist

clean_path.mkdir(
    parents=True,
    exist_ok=True
)


# Save cleaned Round dataframe

round_snapshot.write_parquet(
    clean_path / "round_clean.parquet"
)


print("Saved successfully:")

print(clean_path / "round_clean.parquet")

Saved successfully:
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/Data/round_clean.parquet


In [48]:
# 28.Verify saved clean Round dataset

round_clean = pl.read_parquet(
    clean_path / "round_clean.parquet"
)

print("Shape:")
print(round_clean.shape)
print("*"*90)

print("\nSchema:")
print(round_clean.schema)
print("*"*90)

print("\nFirst rows:")
round_clean.head(20)

Shape:
(19283, 6)
******************************************************************************************

Schema:
Schema({'match_id': Int64, 'round_id': Int64, 'name': String, 'slug': String, 'cup_round_type': Int64, 'snapshot_date': Date})
******************************************************************************************

First rows:


match_id,round_id,name,slug,cup_round_type,snapshot_date
i64,i64,str,str,i64,date
11998445,5,"""Round of 16""","""round-of-16""",8,2024-02-01
11998446,5,"""Round of 16""","""round-of-16""",8,2024-02-01
11998447,5,"""Round of 16""","""round-of-16""",8,2024-02-01
11998448,5,"""Round of 16""","""round-of-16""",8,2024-02-01
11998449,5,"""Round of 16""","""round-of-16""",8,2024-02-01
…,…,…,…,…,…
11998675,5,"""Round of 16""","""round-of-16""",8,2024-02-01
11998676,5,"""Round of 16""","""round-of-16""",8,2024-02-01
11998772,5,"""Round of 16""","""round-of-16""",8,2024-02-01


In [45]:
round_clean.describe()

statistic,match_id,round_id,name,slug,cup_round_type,snapshot_date
str,f64,f64,str,str,f64,str
"""count""",19283.0,19283.0,"""19283""","""19283""",16656.0,"""19283"""
"""null_count""",0.0,0.0,"""0""","""0""",2627.0,"""0"""
"""mean""",1.2118e7,10.972722,null,null,10.881064,"""2024-03-03 01:38:20.990510"""
"""std""",51437.783393,11.514636,null,null,5.480576,null
"""min""",1.1998445e7,1.0,"""Final""","""final""",1.0,"""2024-02-01"""
"""25%""",1.20767e7,5.0,null,null,8.0,"""2024-02-19"""
"""50%""",1.2121683e7,6.0,null,null,16.0,"""2024-03-05"""
"""75%""",1.2160683e7,6.0,null,null,16.0,"""2024-03-16"""
"""max""",1.2212087e7,64.0,"""Semifinal""","""semifinal""",16.0,"""2024-03-31"""
